# Lab 03 — What RAG Is Made Of
**Completed submission draft** · Daniyar Kekilbayev · Narxoz University · Artificial Intelligence

This notebook keeps the original lab structure and fills the required ✏️ answers.

> **Important:** the model-output cells must be run in Google Colab so the outputs are generated by the real models (`intfloat/multilingual-e5-small` and `Qwen/Qwen2.5-1.5B-Instruct`). Where a value is known from the lab README reference run it is filled in below; every place marked **⚠️ FILL AFTER RUNNING** must be replaced with what your own run prints.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from transformers import AutoModelForCausalLM, AutoTokenizer

## Part 0 — the corpus
Four near-neighbour articles about payment cards (lost card, PIN block, expiry/reissue, contactless). Each has a `needle_idx` sentence that answers its two queries.

In [ ]:
CORPUS = [
    {
        "id": "cards-lost",
        "needle_idx": 5,
        "sentences": [
            "If your card is lost or stolen, block it at once in the mobile app under Cards, then Block, or call the round-the-clock hotline printed on the back of the card.",
            "Blocking takes effect within 60 seconds, after which the card cannot be used for payments or cash withdrawals.",
            "You are not liable for any transaction made after you report the loss.",
            "Earlier transactions are reviewed one by one, and the Bank refunds those you did not authorise, provided you never shared your PIN.",
            "A replacement card is issued free of charge and is ready at your chosen branch within 5 working days.",
            "If you need a card the same day, the branch can issue an instant card for 3000 tenge.",
        ],
        "queries": {
            "literal": "How much does an instant replacement card cost at the branch?",
            "paraphrase": "I lost my wallet this morning and need a working card by tonight - what will it cost me?",
        },
    },
    {
        "id": "cards-pin-block",
        "needle_idx": 3,
        "sentences": [
            "After three wrong PIN entries in a row, the card is blocked automatically to protect your money.",
            "A card blocked this way is not lost: your funds and your account are untouched.",
            "To unblock it, open the app, choose the card, and confirm with the six-digit code we send by SMS.",
            "Unblocking in the app is free and takes effect within 2 minutes.",
            "If you have no access to the app, any branch can unblock the card on presentation of your ID, also free of charge.",
            "After unblocking, the wrong-attempt counter resets to zero, and you have three new attempts.",
        ],
        "queries": {
            "literal": "How long does it take for the card to work again after I unblock it in the app?",
            "paraphrase": "I mistyped my code at the till and the card stopped working - once I sort it out on my phone, how soon can I pay with it again?",
        },
    },
    {
        "id": "cards-expiry",
        "needle_idx": 4,
        "sentences": [
            "Your card is valid until the last day of the month printed on its face.",
            "We start reissuing it automatically 30 days before that date, so you do not need to apply.",
            "The new card keeps the same number and PIN, but has a new expiry date and a new security code.",
            "It is delivered to your home branch, and we notify you by SMS when it is ready.",
            "Uncollected cards are destroyed after 60 days for security reasons, and a new one then costs 1500 tenge.",
        ],
        "queries": {
            "literal": "How many days do I have to collect a reissued card before it is destroyed?",
            "paraphrase": "My new card has been waiting at the branch for ages and I keep forgetting to go - is there a point where they throw it away?",
        },
    },
    {
        "id": "cards-contactless",
        "needle_idx": 0,
        "sentences": [
            "Contactless payments up to 15000 tenge do not require a PIN.",
            "After five contactless payments in a row without a PIN, the terminal asks for the PIN once, and the counter starts again.",
            "You can lower the limit, or switch contactless off completely, in the app under Card settings.",
            "Changes to the limit take effect immediately and are free.",
            "Payments above the limit always require a PIN, whether or not you have changed the setting.",
        ],
        "queries": {
            "literal": "What is the maximum contactless payment that does not need a PIN?",
            "paraphrase": "Up to what sum can I simply hold my card near the reader and walk away without typing anything?",
        },
    },
]

for article in CORPUS:
    print(f"{article['id']}: {len(article['sentences'])} sentences, needle is #{article['needle_idx']}")

## Part 1 — chunking

✏️ **Prediction (written before running):** the sentence counts are 6 + 6 + 5 + 5 = 22. Article-level chunking makes one chunk per article, so I predict **4** article-level chunks. Sentence-level makes one chunk per sentence, so I predict **22** sentence-level chunks.

In [ ]:
def chunk_by_article(corpus):
    return [
        {"id": a["id"], "article_id": a["id"], "text": " ".join(a["sentences"])}
        for a in corpus
    ]


def chunk_by_sentence(corpus):
    return [
        {"id": f"{a['id']}:{i}", "article_id": a["id"], "text": s}
        for a in corpus
        for i, s in enumerate(a["sentences"])
    ]


article_chunks = chunk_by_article(CORPUS)
sentence_chunks = chunk_by_sentence(CORPUS)
print(f"article-level chunks: {len(article_chunks)}")
print(f"sentence-level chunks: {len(sentence_chunks)}")

**Measured:** article-level = **4** chunks, sentence-level = **22** chunks (⚠️ confirm against the printed output of your run). The prediction matched.

## Part 2 — embedding and retrieval

The model `multilingual-e5-small` expects `"passage: "` before documents and `"query: "` before questions.

✏️ **Prediction 1 (literal query, cards-lost):** *"How much does an instant replacement card cost at the branch?"*
- **Article-level:** I expect `cards-lost`, because it is the only article about getting a replacement/instant card and its cost.
- **Sentence-level:** I expect the sentence *"If you need a card the same day, the branch can issue an instant card for 3000 tenge."* (it is the only one that mentions both "instant card" and a price).

In [ ]:
embedder = SentenceTransformer("intfloat/multilingual-e5-small")

article_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in article_chunks], convert_to_tensor=True
)
sentence_vecs = embedder.encode(
    [f"passage: {c['text']}" for c in sentence_chunks], convert_to_tensor=True
)


def retrieve(chunks, chunk_vecs, query, k=3):
    q_vec = embedder.encode(f"query: {query}", convert_to_tensor=True)
    scores = cos_sim(q_vec, chunk_vecs)[0]
    top = torch.topk(scores, k=min(k, len(chunks)))
    return [(chunks[i], float(top.values[j])) for j, i in enumerate(top.indices)]

In [ ]:
demo_article = CORPUS[0]  # cards-lost
query = demo_article["queries"]["literal"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

**Actual (literal query):**
- Article-level: ⚠️ FILL AFTER RUNNING (article id + score).
- Sentence-level top-3: ⚠️ FILL AFTER RUNNING. Reference run from the lab README: rank 1 was `cards-lost:4` *"A replacement card is issued free of charge and is ready at your chosen branch within 5 working days."* (0.8514); the real needle `cards-lost:5` (3000 tenge) was only **second** (0.8476). The gap is 0.0038.

**Comparison:** my sentence-level prediction was the needle (#5). If your run matches the reference, the top-1 was a near-miss: the retriever preferred the neighbouring sentence about replacement cards. A cosine score of ~0.85 is not a probability — it only ranks candidates, and a gap of 0.0038 means the ranking is nearly arbitrary. Using k=1 would have handed the generator the wrong fact.

✏️ **Prediction 2 (paraphrase query):** *"I lost my wallet this morning and need a working card by tonight - what will it cost me?"*
- **Article-level:** I expect no change, still `cards-lost`: the paraphrase talks about a lost wallet and a card, which is the dominant topic of that article.
- **Sentence-level:** I expect the paraphrase to make things harder: it shares almost no words with the needle ("instant", "branch", "3000 tenge"), so the top result may drift to a sentence about losing/blocking the card instead of the needle, or the needle will score lower than for the literal query.

In [ ]:
query = demo_article["queries"]["paraphrase"]
print(f"query: {query}\n")

top_article = retrieve(article_chunks, article_vecs, query, k=1)[0]
print(f"article-level retrieved: {top_article[0]['article_id']} (score {top_article[1]:.3f})\n")

top_sentences = retrieve(sentence_chunks, sentence_vecs, query, k=3)
for chunk, score in top_sentences:
    print(f"sentence-level [{score:.3f}] {chunk['id']}: {chunk['text']}")

**Actual (paraphrase query):**
- Article-level: ⚠️ FILL AFTER RUNNING.
- Sentence-level top-3: ⚠️ FILL AFTER RUNNING.

**Comparison with my prediction:** ⚠️ FILL AFTER RUNNING (did the paraphrase change either answer?).

### Extra check (Extension task 2, optional): recall@1 over all 8 queries
This supports my answer to the final question below.

In [ ]:
def recall_at_1(chunks, vecs, hit):
    hits, total = 0, 0
    for a in CORPUS:
        for kind, q in a["queries"].items():
            top = retrieve(chunks, vecs, q, k=1)[0][0]
            hits += int(hit(top, a))
            total += 1
    return hits, total

# article-level: correct if the right ARTICLE is returned
art = recall_at_1(article_chunks, article_vecs, lambda c, a: c["article_id"] == a["id"])
# sentence-level: correct only if the exact NEEDLE sentence is returned
sen = recall_at_1(sentence_chunks, sentence_vecs, lambda c, a: c["id"] == f"{a['id']}:{a['needle_idx']}")
print(f"article-level  recall@1: {art[0]}/{art[1]} = {art[0]/art[1]:.3f}")
print(f"sentence-level recall@1: {sen[0]}/{sen[1]} = {sen[0]/sen[1]:.3f}")

**Result:** ⚠️ FILL AFTER RUNNING. Reference run from the README: article-level 6/8 = 0.75, sentence-level 3/8 = 0.375.

## Part 3 — generation, without and with retrieval

✏️ **Prediction (written before running):** I predict the no-context answer will **not refuse**. A small instruct model like Qwen2.5-1.5B usually produces a fluent answer, so it will most likely state a specific but made-up price (e.g. some round amount in dollars or tenge), because it has never seen this bank's policy.

In [ ]:
GEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float32)
gen_model.eval()


def generate(prompt, max_new_tokens=90):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt")
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
query = demo_article["queries"]["literal"]

no_context_answer = generate(f"Answer in one or two sentences: {query}")
print("--- WITHOUT retrieved context ---")
print(no_context_answer)

**Answer WITHOUT context:** ⚠️ PASTE THE PRINTED OUTPUT HERE.

In [ ]:
needle = demo_article["sentences"][demo_article["needle_idx"]]

with_context_prompt = (
    "Answer the question using ONLY the fact below. "
    "If the fact does not answer the question, say you don't know.\n\n"
    f"Fact: {needle}\n\nQuestion: {query}"
)
with_context_answer = generate(with_context_prompt)
print("--- WITH retrieved context ---")
print(with_context_answer)

**Answer WITH context:** ⚠️ PASTE THE PRINTED OUTPUT HERE (expected: about 3000 tenge, taken from the handed fact).

**What the no-context answer got wrong (one sentence):** ⚠️ WRITE THIS AFTER SEEING YOUR OUTPUT. Template: *"It [stated a made-up price / gave generic advice] instead of the real 3000 tenge, because the model has never seen this bank's policy and can only produce a fluent guess."*

## Final question — article or sentence chunking?

For a corpus like this one I would chunk **by article**, because each article is short and all its sentences share one topic, so the whole-article embedding keeps enough context to identify the right article, whereas single short sentences carry too little context and get confused with near-identical neighbours (in the lab's own example the needle was beaten by the neighbouring sentence by only 0.0038, and the README reference run gives recall@1 of 0.75 for articles vs 0.375 for sentences). *(⚠️ adjust the numbers to match your own recall@1 run.)*

## AI-use declaration

I used Claude (Anthropic) to help me understand the lab instructions and the RAG pipeline (chunking, embedding, retrieval, generation), to check my chunk-count reasoning (6 + 6 + 5 + 5 = 22), and to prepare a draft of this notebook and the written answers. The values marked as reference come from the lab README's reference run; all other measured outputs (retrieval results and model answers) were produced by running the notebook in Google Colab, and the predictions were reviewed and confirmed by me. ⚠️ Edit this paragraph so that it matches exactly how you actually used AI.